# `MapReduce`

* Le paradigme `MapReduce` est un passage obligatoire pour tout Dév Big Data qui se respecte ! 
* Pour vous aider à mieux comprendre ce paradigme : une [petite vidéo](https://www.youtube.com/watch?v=QNB1SZm2jS4&list=PLcHc4QczJB9MADOyOIDV4ERP9uTGBhgHf&index=14) ainsi que deux articles [michel-noll](https://www.michael-noll.com/tutorials/writing-an-hadoop-mapreduce-program-in-python/), [dev.to](https://dev.to/boyu1997/run-python-mapreduce-on-local-docker-hadoop-cluster-1g46)
* **WordCount** est l'application proposée dans [l'article de Google](https://1drv.ms/b/s!AmJGbSlW18YGsdMQRK0WGJGPcYCO9A?e=UOQyRx) et consiste en un comptage d'occurence de mots en mode distribué à partir d'un ensemble de fichiers (un corpus). 
* Ce qui peut répondre à la question métier suivante : _Déterminer les mots les plus utilisés dans la langue française (à partir des doc de la BNF par ex)_. Nous allons le réaliser avec le code Python classique et en mode distribué avec `YARN` et avec `Spark`. 
<center><img src="https://big-data.developpez.com/tutoriels/apprendre-faire-choix-architecture-big-data/images/image-18.png"></center>

* Autre exemple de pb à résoudre avec `MapReduce` : 
    - Soit une société, appelons-la Ntfx., qui propose des films et séries télévisées (_200 000 dans le catalogue_) sur Internet (_50 millions de clients_). Toute ressemblance avec Netflix est évidemment fortuite.
    - Supposons que chaque client a visionné en moyenne 20 films, qu’il a notés. Ntfx. veuille calculer pour chaque film la moyenne des notes. 
    - Cette tâche est très simple comparée à d’autres tâches, comme de trouver des proximités de goût entre clients. Ce besoin de calculs est néanmoins primordial et doit être fait régulièrement 
    - Formalisons légèrement le problème :
        - En Input, ns avons 20 × 50 millions = 1 milliard d’enregistrements de la forme (Serge Tournesol, Manhattan, 4.5), càd, « Serge Tournesol a donné la note de 4.5 (sur 5) au film Manhattan ».
        - En Output, ns voulons les moyennes des 200 000 enregistrements de la forme (Manhattan, 4.1), càd, « La moyenne du film Manhattan est de 4.1 parmi les clients de Ntfx. ».<br>
        <br>
    - **Avec un milliard d’enregistrements, le pb est qu'avec un seul ordinateur le programme de calcul de la moyenne prendrait énormément de temps. Nous allons alors partager le travail entre un grand nombre d'ordi : les ordi-mappers et les ordi-reducers**

* Autre exemple de pb : friend recommendation using `MapReduce` : voir <http://andresromero.github.io/People-you-may-know/>

# WordCount Avec `Python` (pur)

* **Q** : Réaliser un 1er WordCount sur le fichier [helloHadoopDocker.txt](https://1drv.ms/t/s!AmJGbSlW18YGsdMaKCY6r9X8otlw3A?e=gKdFZY) (seul) et un 2ème WordCount sur le corpus de fichiers : [helloHadoop.txt](https://1drv.ms/t/s!AmJGbSlW18YGsdMSlQFVhnoU8B5WcQ?e=J2pxDw) et [helloDocker.txt](https://1drv.ms/t/s!AmJGbSlW18YGsdMRf3HiQfSUvRFfZA?e=KVnbUe) (ensemble)

* **Correc** : WordCount sur le le fichier [helloHadoopDocker.txt](https://1drv.ms/t/s!AmJGbSlW18YGsdMTQAWOfCUqVs7Ogg?e=mZUOXx) (seul)

In [2]:
import os
# os.chdir(r'/')
# os.chdir(r'/mnt/c/Users/bejao/OneDrive/data/')
os.chdir(r'C:/Users/bejao/OneDrive/data/')
# os.chdir(r'/home/sayf/hadoop/')
os.getcwd()

'C:\\Users\\bejao\\OneDrive\\data'

In [4]:
with open('helloHadoopDocker.txt', 'r') as f :
    f_content = f.read()   # On stocke tt le contenu du fichier ds 1str.
f_content

words = f_content.split()
words

counts = {}
for word in words:
    if word in counts:
        counts[word] += 1
    else:
        counts[word] = 1

print(counts)

{'Hello': 2, 'Hadoop': 1, 'Docker': 1}


* **Correc** : WordCount sur le corpus de fichiers : [helloHadoop.txt](https://1drv.ms/t/s!AmJGbSlW18YGsdMSlQFVhnoU8B5WcQ?e=J2pxDw), [helloDocker.txt](https://1drv.ms/t/s!AmJGbSlW18YGsshpUurPNpclLKn87A?e=n1hSMM) (ensemble)

In [6]:
corpus = os.listdir('./wordcount')
corpus
# ['helloDocker.txt', 'helloHadoop.txt', 'helloHadoopDocker.txt']

counts = {}
for item in corpus : 
    with open('./wordcount/'+item, 'r') as f :
        f_content = f.read()   # On stocke tt le contenu du fichier ds 1str.
    words = f_content.split()   
    print(words) 
    for word in words:
        if word in counts:
            counts[word] += 1
        else:
            counts[word] = 1

print(counts)

['Hello', 'Docker']
['Hello', 'Hadoop']
['Hello', 'Hadoop', 'Hello', 'Docker']
{'Hello': 4, 'Docker': 2, 'Hadoop': 2}


* **Q** : Lire le fichier [README](https://1drv.ms/u/s!AmJGbSlW18YGr493FzdyZPm-Y6pJ7A) de `Spark` et compter le nombre de mots et le nombre de ligne qui commence par 'a' (par 'b') ?
* **Correc** : L'idée est que nous pouvons introduire des `if`, des calculs très spécifiques : moy, médiane,... , max 

In [40]:
!cat spark-readme.md | more

# Apache Spark

Spark is a fast and general cluster computing system for Big Data. It provides
high-level APIs in Scala, Java, Python, and R, and an optimized engine that
supports general computation graphs for data analysis. It also supports a
rich set of higher-level tools including Spark SQL for SQL and DataFrames,
MLlib for machine learning, GraphX for graph processing,
and Spark Streaming for stream processing.

<http://spark.apache.org/>


## Online Documentation

You can find the latest Spark documentation, including a programming
guide, on the [project web page](http://spark.apache.org/documentation.html).
This README file only contains basic setup instructions.

## Building Spark

Spark is built using [Apache Maven](http://maven.apache.org/).
To build Spark and its example programs, run:

m--More--

In [45]:
# Lire les lignes du readme (pr vérification)
# with open('spark-readme.md', "r") as f :
# 	for line in f :
# 		print(line)

# Stocker tt le contenu du fichier ds 1str
with open('spark-readme.md', "r") as f :
	f_content = f.read()

words = f_content.split('\n')

liste_a = []
for item in words :
	if 'a' in item :
		liste_a.append(item)

liste_a
# liste_a = [i for i in liste if 'a' in i]

## Méthode 1 : passer par des listes de compréhension
# with open(readme, "r") as f :
# 	f_content = f.readlines() 	# Tu stocke tt le contenu du fichier ds une liste.

# liste_a = [i for i in f_content if 'a' in i]
# liste_b = [i for i in f_content if 'b' in i]
# len(liste_a)

['# Apache Spark',
 'Spark is a fast and general cluster computing system for Big Data. It provides',
 'high-level APIs in Scala, Java, Python, and R, and an optimized engine that',
 'supports general computation graphs for data analysis. It also supports a',
 'rich set of higher-level tools including Spark SQL for SQL and DataFrames,',
 'MLlib for machine learning, GraphX for graph processing,',
 'and Spark Streaming for stream processing.',
 '<http://spark.apache.org/>',
 '## Online Documentation',
 'You can find the latest Spark documentation, including a programming',
 'guide, on the [project web page](http://spark.apache.org/documentation.html).',
 'This README file only contains basic setup instructions.',
 '## Building Spark',
 'Spark is built using [Apache Maven](http://maven.apache.org/).',
 'To build Spark and its example programs, run:',
 '    build/mvn -DskipTests clean package',
 '(You do not need to do this if you downloaded a pre-built package.)',
 'You can build Spark u

# Les fonctions `Map` et `Reduce` native de `Python`

## `Map` [help1](https://stackabuse.com/map-filter-and-reduce-in-python-with-examples/), [help2](https://goutomroy.medium.com/python-lambda-map-filter-reduce-261f29561fcc)

* `Map` prend en argument une fonction `lamdba` et une collection et retourne une collection. La fonction étant appliquée sur chaque élément de la collection. 
* Executer le code ci-dessous et expliquer ce qu'il fait. Vous rappelez vous de ce qu'est une fonction anonyme? 

In [9]:
collec = [1, 2, 4, 5, 6]
res = map(lambda x: 2*x, collec)
res
for i in res:
    print(i)

2
4
8
10
12


* Créer une liste de nombres entiers aléatoires contenant 1000 exemples. 
* A l'aide de la fonction `map`, retourner une collection qui contient `True` si le nombre est pair, `False` sinon.

## `Reduce`

* La fonction `reduce` prend en entrée une collection et retourne une reduction de celle ci. Par exemple, elle permet de faire la somme des éléments d'une liste. 
Executer le code suivant.

In [10]:
from functools import reduce
a = [1, 2, 3, 5]
sum_a = reduce(lambda x,y: x+y, a)
print(sum_a)

11


* On considère le big dataset suivant, on cherche à l'aide des fonctions `Map` et `Reduce` de compter le nombre de mots total ? 

In [ ]:
a = ["ceci n'est pas du big data", 'bonjour voila du texte', "mangez donc un croissant", 'vivement la pause']

# WordCount Avec `Hadoop` (`YARN`)

* **Q : Coder en `Java` un job MapReduce sur les derniers fichiers et récupérer les résultats en local ?**  
* **Q : Coder en `Python` un job MapReduce sur les derniers fichiers et récupérer les résultats en local ?**  

# **Correc** : 

* Q : Démarrer/Arrêter [YARN](https://www.lebigdata.fr/yarn-apache-hadoop) ? 
* Rép : 
    - `Yarn web page` (Resource manager): http://localhost:8088/cluster
    - `NameNode` et `YARN ResourceManager` constitue maintenant ce que ns appellons `node-master` et les `DataNode` et `YARN NodeManager` sont appelés node1, node2 et node3.

In [1]:
# !sbin/start-yarn.sh
# !sbin/stop-yarn.sh
!jps


5568 Jps
5060 ResourceManager
4853 DataNode
3834 NameNode
5387 NodeManager
4603 DataNode
4716 DataNode


In [26]:
# !cd /home/sayf/hadoop
!hdfs dfs -ls /hadoop

/bin/bash: bin/hdfs: No such file or directory


In [31]:
# Déposer les fichiers ds Hadoop
# !pwd
# !cd home/sayf/hadoop
!hdfs dfs -put /mnt/c/Users/bejao/OneDrive/data/helloHadoopDocker.txt /mnt/c/Users/bejao/OneDrive/data/wordcount /hadoop 
# !bin/hdfs dfs -ls /hadoop

/
put: `/hadoop/helloHadoopDocker.txt': File exists
put: `/hadoop/wordcount/helloDocker.txt': File exists
put: `/hadoop/wordcount/helloHadoop.txt': File exists
put: `/hadoop/wordcount/helloHadoopDocker.txt': File exists


# Avec `Java`

* Supp le dossier output : [stackoverflow](https://stackoverflow.com/questions/31817186/mapreduce-on-hadoop-says-output-file-already-exists)

In [18]:
bin/hdfs dfsadmin -safemode leave
bin/hdfs dfs -rm -r /user/sayf/output
# !hdfs dfs -ls /user/sayf/

/bin/bash: bin/hdfs: No such file or directory


In [36]:
# Lancer le job
yarn jar /home/sayf/hadoop/share/hadoop/mapreduce/hadoop-mapreduce-examples-2.7.3.jar wordcount /hadoop/wordcount /user/sayf/output


22/01/27 20:05:46 INFO client.RMProxy: Connecting to ResourceManager at /127.0.0.1:8032
22/01/27 20:05:47 INFO input.FileInputFormat: Total input paths to process : 3
22/01/27 20:05:47 INFO mapreduce.JobSubmitter: number of splits:3
22/01/27 20:05:47 INFO mapreduce.JobSubmitter: Submitting tokens for job: job_1643283319756_0004
22/01/27 20:05:47 INFO impl.YarnClientImpl: Submitted application application_1643283319756_0004
22/01/27 20:05:47 INFO mapreduce.Job: The url to track the job: http://DESKTOP-G4OOFUM.localdomain:8088/proxy/application_1643283319756_0004/
22/01/27 20:05:47 INFO mapreduce.Job: Running job: job_1643283319756_0004
22/01/27 20:05:53 INFO mapreduce.Job: Job job_1643283319756_0004 running in uber mode : false
22/01/27 20:05:53 INFO mapreduce.Job:  map 0% reduce 0%
22/01/27 20:06:00 INFO mapreduce.Job:  map 33% reduce 0%
22/01/27 20:06:01 INFO mapreduce.Job:  map 67% reduce 0%
22/01/27 20:06:02 INFO mapreduce.Job:  map 100% reduce 0%
22/01/27 20:06:06 INFO mapreduce.Jo

In [37]:
# Réccupérer la sortie

hdfs dfs -ls /user/sayf/output
hdfs dfs -cat /user/sayf/output/part-r-00000

Docker	2
Hadoop	2
Hello	4


## Remarques importantes en vrac


* Après avoir soumis un job MR, Yarn affiche beaucoup de lignes. Repérez les suivantes :  
    - `yyyy-mm-dd hh:mn:ss [main] input.FileInputFormat: Total input paths to process : 1`
    - `yyyy-mm-dd hh:mn:ss [main] mapreduce.JobSubmitter: number of splits:1`  
    <br>
* Ces lignes indiquent que vos données sont composées d’un seul fichier et que du coup, 2è ligne, il ne lance qu’un seul Mapper.

* La ligne suivante donne l’identifiant de l’application :
    - `yyyy-mm-dd ... impl.YarnClientImpl: Submitted application application_1515425519656_0423`

* Le mode « uber » est une optimisation quand les données sont très petites : il ne lance qu’un seul Mapper, sur une seule machine.
    - `yyyy-mm-dd mapreduce.Job: Job job_1515425519656_0423 running in uber mode : true`

* Il est tt à fait possible que plusieurs machines participent à la réduction, on peut le savoir avec le nb de fichiers part-r-00000, part-r-00001,. . .

* vous aurez un état d’avancement des tâches map et reduce :
    - `yyyy-mm-dd 10:44:11,359 INFO [main] mapreduce.Job: map 100% reduce 100%`

* À la fin du job, vous allez voir ces informations très importantes, le nombre de paires (clé, valeur) en entrée et en sortie du map :
Map-Reduce Framework
    Map input records=30067
    Map output records=28761

* Par exemple, ici, il y a eu 30067 lignes traitées et seulement 28761 ont généré une paire à destination du `reducer`. 
* Les autres lignes ont donc été rejetées, soit par un test, soit par une exception.
* Quand `Map output records` vaut zéro, c’est sûrement que votre mapper ne parvient pas à travailler du
* tout ; vous aurez à analyser les logs sur Yarn !

* Il y a ensuite le nombre de paires (clé, valeur) traitées par le combiner, ici aucune car pas de combiner :
    Combine input records=0
    Combine output records=0

* Pour finir, il y a le nombre de clés différentes ainsi que les paires traitées par le reducer :
    Reduce input groups=2
    Reduce input records=28761
    Reduce output records=2
* Ici, il y a deux clés distinctes parmi les (clé, valeur) produites par les mappers, 28761 paires en tout en entrée et seulement 2 en sortie (après le reduce).

* Pour tuer un job MR qui n’en finit pas, CTRL-C ne suffit pas. En fait, le job continue en arrière-plan.
* Pour la tuer vraiment, il faut avoir son identifiant, par exemple application_1455868731302_0020 et taper : 
`yarn application -kill` suivi de son identifiant.
* On peut obtenir l’identifiant soit avec l’interface Web, soit dans les premières lignes affichées lors du lancement

# Avec `Python`

* Hadoop permet de programmer un Job `MapReduce` dans d’autres langages que `Java` : `Ruby`, `Python`, `C++`, ...
- En fait, il suffit que le langage utilisé permet de lire `stdin` et d'écrire ses résultats sur `stdout`.
- Le Mapper est un programme qui lit des lignes sur `stdin`, calcule ce qu’il veut, puis écrit des lignes au format `"%s\t%s\n"` (clé, valeur) sur la sortie
* Pour lancer un job MR en Python, il faut : 
    - Trouver le `jar` `hadoop-streaming.jar` avec la commande suivante : `find / -name hadoop-streaming*.jar`
    - Préparer les deux scripts mapper.py et reducer.py et les déposer sur HDFS
    - Taper la commande suivante :
        <br>
        ```
        hdfs dfsadmin -safemode leave   # optionnelle 
        yarn jar $HADOOP_HOME/share/hadoop/tools/lib/hadoop-streaming-2.7.3.jar \ 
        -mapper "python3 mapper.py" -reducer "python3 reducer.py" \
        -input path_to_input_folder -output output
        ```
        <br>
* Il est possible que le `hadoop-streaming.jar` se trouve dans (à vérifier) : 
    - `/usr/lib/hadoop-mapreduce/hadoop-streaming.jar`  # autre path possible 
    - `/opt/hadoop-2.7.4/share/hadoop/tools/lib/hadoop-streaming-2.7.4.jar` 
* Vous pouvez indiquer pour l'`output` un path spécifique `/user/hadoop/output-N`  
        <br>
* Voici le « WordCount » programmé en Python :


In [ ]:
# mapper.py
import sys
# traiter chaque ligne de l'entrée standard
for ligne in sys.stdin:
# couper en mots et traiter chacun d'eux
    for mot in ligne.split():
        paire = (mot, 1)
        # écrire la paire
        print('%s\t%s' % paire)     

* Concernant le `reducer.py`, Vous allez recevoir N lignes composées de paires (clé,valeur). Vous devez accumuler les valeurs correspondant à des clés identiques (somme, moyenne, min, max. . . ) 
* Le parcours se fait ligne par ligne. Il faut donc mémoriser la clé de la ligne précédente ainsi que l’accumulation des valeurs de cette clé. Quand la ligne courante a la même clé que la précédente, on met à jour le cumul. Sinon, on affiche le cumul (et sa clé) sur la sortie et on le remet à zéro. 
* À la fin, ne pas oublier d’afficher la dernière clé et le cumul.


In [ ]:
# reducer.py
import sys
cle_prec, nombre_total = None, 0
for ligne in sys.stdin:
    cle, valeur = ligne.split('\t', 1)
    if cle == cle_prec:
        nombre_total += int(valeur)
    else:
        if cle_prec:
            paire = (cle_prec, nombre_total)
            print('%s\t%s' % paire)
        cle_prec = cle
        nombre_total = int(valeur)
if cle_prec == cle:
    paire = (cle_prec, nombre_total)
    print('%s\t%s' % paire)

In [ ]:
import sys

cle_prec, nombre_total = None, 0
output = {}
for ligne in sys.stdin:
    cle, valeur = ligne.split('\t', 1)
    if cle in output:
        output[cle] += int(valeur.strip())
    else:
        output[cle] = int(valeur.strip())
for key, val in output.items():
    print('%s\t%s' % (key, val))

In [ ]:
# MapReduce dans HDFS

# On supp le dossier de sortie
bin/hdfs dfs -rm -r /user/sayf/output

22/01/27 20:34:05 INFO fs.TrashPolicyDefault: Namenode trash configuration: Deletion interval = 1440 minutes, Emptier interval = 0 minutes.
Moved: 'hdfs://localhost:8020/user/sayf/output' to trash at: hdfs://localhost:8020/user/sayf/.Trash/Current


In [ ]:
~/hadoop/bin/yarn jar ~/hadoop/share/hadoop/tools/lib/hadoop-streaming-2.7.3.jar \
        -mapper "python3 /mnt/c/Users/bejao/OneDrive/BigData/2-TP-MapReduce-Python-Yarn-Spark/mapper-wordcount.py" \
        -reducer "python3 /mnt/c/Users/bejao/OneDrive/BigData/2-TP-MapReduce-Python-Yarn-Spark/reducer-wordcount.py" \
        -input /hadoop/wordcount/helloHadoopDocker.txt -output output

packageJobJar: [/tmp/hadoop-unjar8445944579881732874/] [] /tmp/streamjob6841270555582387362.jar tmpDir=null
22/01/27 20:34:09 INFO client.RMProxy: Connecting to ResourceManager at /127.0.0.1:8032
22/01/27 20:34:09 INFO client.RMProxy: Connecting to ResourceManager at /127.0.0.1:8032
22/01/27 20:34:09 INFO mapred.FileInputFormat: Total input paths to process : 1
22/01/27 20:34:10 INFO mapreduce.JobSubmitter: number of splits:2
22/01/27 20:34:10 INFO mapreduce.JobSubmitter: Submitting tokens for job: job_1643283319756_0007
22/01/27 20:34:10 INFO impl.YarnClientImpl: Submitted application application_1643283319756_0007
22/01/27 20:34:10 INFO mapreduce.Job: The url to track the job: http://DESKTOP-G4OOFUM.localdomain:8088/proxy/application_1643283319756_0007/
22/01/27 20:34:10 INFO mapreduce.Job: Running job: job_1643283319756_0007
22/01/27 20:34:16 INFO mapreduce.Job: Job job_1643283319756_0007 running in uber mode : false
22/01/27 20:34:16 INFO mapreduce.Job:  map 0% reduce 0%
22/01/27 2

In [ ]:
# Réccupérer la sortie

# !hdfs dfs -ls /user/sayf/output

!bin/hdfs dfs -cat /user/sayf/output/part-00000

Docker	1
Hadoop	1
Hello	2


# `MapReduce` avec `Yarn` : Bonnes pratiques

* Il est intéressant de voir qu’on peut lancer des job MapReduce en local dans un tube Unix, mais en mode séquentiel :
<br>
    ```
    cat data | mapper.py | sort | reducer.py
    head -50 data.txt > test.txt 
    cat test.txt | python3 mapper.py | sort | python reducer.py
    cat test.txt | python3 mapper.py | sort -k1, 1 | python reducer.py
    ```

In [1]:
!cat 'helloHadoopDocker.txt' 
# | python3 ../BigData/2-TP-MapReduce-Python-Yarn-Spark/mapper-wordcount.py | sort | python3 ../BigData/2-TP-MapReduce-Python-Yarn-Spark/reducer-wordcount.py

# !cat 'wordcount/helloHadoopDocker.txt' 'wordcount/helloHadoop.txt' 'wordcount/helloDocker.txt' | python3 ../BigData/2-TP-MapReduce-Python-Yarn-Spark/mapper-wordcount.py | sort | python3 ../BigData/2-TP-MapReduce-Python-Yarn-Spark/reducer-wordcount.py

# Q : Comment passer tt un folder ? 
# https://stackoverflow.com/questions/44899209/python-how-to-pass-a-directory-as-mapreduce-input
# !cat wordcount/*.txt | python3 ../BigData/2-TP-MapReduce-Python-Yarn-Spark/mapper-wordcount.py | sort | python3 ../BigData/2-TP-MapReduce-Python-Yarn-Spark/reducer-wordcount.py

cat: helloHadoopDocker.txt: No such file or directory


# `MapReduce` avec `Spark`

In [ ]:
!spark/sbin/start-history-server.sh

## Réduire la verbosité de `Spark`

* Par défaut `Spark` émet trop de logs, Si vous souhaitez supprimer les nombreuses lignes d’information qui précèdent la réponse, vous pouvez réduire la quantité de logs émis en configurant `log4j`. Pour cela, copiez `log4j.properties.template` en `log4j.properties` et, dans ce dernier fichier, remplacez ensuite la ligne `log4j.rootCategory=INFO`, console par`log4j.rootCategory=ERROR`, console. 
* Seuls les logs d'un niveau de criticité supérieur ou égal à `ERROR` seront alors affichés dans la console.
* [Stackoverflow](https://stackoverflow.com/questions/27781187/how-to-stop-info-messages-displaying-on-spark-console)

## Démarrer le `Master`

`UI` [Spark Master](http://localhost:8080/
), `UI` [Spark Worker](http://localhost:8081/), Tuto pour les diff `UI` de `Spark` : [SparkbyExamples](https://sparkbyexamples.com/spark/spark-web-ui-understanding/)

In [3]:
# !pwd
import os
os.chdir('/home/sayf/hadoop')

In [1]:
!spark/sbin/start-master.sh
# se connecter au http://localhost:8080/

# !spark/sbin/stop-master.sh


# !spark/sbin/start-all.sh
# start-all.sh lance le NN et Yarn ; contrairement à son nom
#  Ns pouvons se connecter au localhost:50070 (NN) et localhost:50070 (Yarn)
# Warning : This script is Deprecated. Instead use start-dfs.sh and start-yarn.sh (qui se trouvent ds le dossier /hadoop/sbin)

# !spark/sbin/stop-all.sh  

# sbin/stop-dfs.sh and sbin/stop-yarn.sh


# Si vs rencontrez une erreur de type start-all.sh command not found, exécuter la commande suivante
# !chmod -R 755 spark
# !chmod +x spark/bin/start-worker.sh

# !sudo chmod +x spark/sbin

In [1]:
!jps

25300 SecondaryNameNode
25816 NodeManager
25066 DataNode
26060 Jps
24860 NameNode
25471 ResourceManager


## Connecter un `Worker` au `Master`

In [32]:
!spark/sbin/start-worker.sh spark://DESKTOP-G4OOFUM.localdomain:7077
# se connecter au http://localhost:8081/

# !spark/sbin/stop-worker.sh spark://DESKTOP-G4OOFUM.localdomain:7077

localhost: org.apache.spark.deploy.worker.Worker running as process 8983.  Stop it first.


# Configurer les paramètres de la Session `Spark`

In [1]:
from pyspark import SparkConf
from pyspark import SparkContext
from pyspark.sql import SparkSession

In [2]:
app_name = "spark_readme"
master = "spark://DESKTOP-G4OOFUM.localdomain:7077"
nb_cores = 3
# parallélisme = 3
# memory = 3

In [51]:
conf = SparkConf()
conf = conf.setAppName(app_name + " via conf")
conf = conf.setMaster(master)
# conf = conf.set("spark.serialize", "org.apache.spark.serializer.kryoSerializer")
# conf = conf.set("spark.cores.max", " %s" %(nb_cores))
# conf = conf.set("spark.executor.memory", " %sg" %memory)
# conf = conf.set("spark.driver.memory", " %sg" %memory)
# conf = conf.set("spark.kryoserializer.buffer.max", "1024m")
# conf = conf.set("spark.driver.maxResultSize", "10g")
# conf = conf.set("spark.cores.max", " %s" %(nb_cores))
# conf = conf.set("spark.default.parallelism", " %s" %(nb_cores * parallélisme))

In [58]:
# Instantiation d'un SparkContext
# le spark context détermine les ressources disponibles pour l'application
sc.stop()
sc = SparkContext(conf = conf).getOrCreate()
spark = SparkSession.builder.config(conf = conf).getOrCreate()
sc

<SparkContext master=spark://DESKTOP-G4OOFUM.localdomain:7077 appName=spark_readme via conf>

# WordCount avec `PySpark`

In [23]:
# Lecture d'un fichier texte : le fichier est décomposé en lignes.
lines = sc.textFile("file:///mnt/c/Users/bejao/OneDrive/data/helloHadoopDocker.txt")

# flatMap() : Décomposition de chaque ligne en mots
# map() : Chacun des mots est transformé en une clé-valeur
# reduceByKey() : Les valeurs associées à chaques clé sont sommées
# collect() : Le résultat est récupéré

word_counts = lines.flatMap(lambda line:line.split(' ')).map(lambda word: (word, 1)).reduceByKey(lambda count1, count2: count1 + count2).collect()

# Chaque paire (clé, valeur) est affichée
for (word, count) in word_counts:
    print(word, count)

Hello 2
Hadoop 1
Docker 1


# Q : Lancer un Job MapReduce qui consiste : 

* à compter le nombre de ligne où apparaisssent les caractères ‘a’ et ‘b’
* à compter le nombre de mots
* à afficher les mots de plus de trois lettres les plus fréquents (ds le `readme` de `Spark`)
* [help Stackoverflow](https://stackoverflow.com/questions/23280629/multiple-sparkcontexts-error-in-tutorial)

In [47]:
!cat /mnt/c/Users/bejao/OneDrive/data/spark-readme.md

# Apache Spark

Spark is a fast and general cluster computing system for Big Data. It provides
high-level APIs in Scala, Java, Python, and R, and an optimized engine that
supports general computation graphs for data analysis. It also supports a
rich set of higher-level tools including Spark SQL for SQL and DataFrames,
MLlib for machine learning, GraphX for graph processing,
and Spark Streaming for stream processing.

<http://spark.apache.org/>


## Online Documentation

You can find the latest Spark documentation, including a programming
guide, on the [project web page](http://spark.apache.org/documentation.html).
This README file only contains basic setup instructions.

## Building Spark

Spark is built using [Apache Maven](http://maven.apache.org/).
To build Spark and its example programs, run:

    build/mvn -DskipTests clean package

(You do not need to do this if you downloaded a pre-built package.)

You can build Spark using more than one thread by using the -T option with Maven,

In [62]:
readme_rdd = sc.textFile('file:///mnt/c/Users/bejao/OneDrive/data/spark-readme.md').cache()
readme_rdd

file:///mnt/c/Users/bejao/OneDrive/data/spark-readme.md MapPartitionsRDD[4] at textFile at NativeMethodAccessorImpl.java:0

In [63]:
readme_rdd.take(5)

['# Apache Spark',
 '',
 'Spark is a fast and general cluster computing system for Big Data. It provides',
 'high-level APIs in Scala, Java, Python, and R, and an optimized engine that',
 'supports general computation graphs for data analysis. It also supports a']

In [64]:
readme_rdd_all = readme_rdd.collect()
readme_rdd_all

['# Apache Spark',
 '',
 'Spark is a fast and general cluster computing system for Big Data. It provides',
 'high-level APIs in Scala, Java, Python, and R, and an optimized engine that',
 'supports general computation graphs for data analysis. It also supports a',
 'rich set of higher-level tools including Spark SQL for SQL and DataFrames,',
 'MLlib for machine learning, GraphX for graph processing,',
 'and Spark Streaming for stream processing.',
 '',
 '<http://spark.apache.org/>',
 '',
 '',
 '## Online Documentation',
 '',
 'You can find the latest Spark documentation, including a programming',
 'guide, on the [project web page](http://spark.apache.org/documentation.html).',
 'This README file only contains basic setup instructions.',
 '',
 '## Building Spark',
 '',
 'Spark is built using [Apache Maven](http://maven.apache.org/).',
 'To build Spark and its example programs, run:',
 '',
 '    build/mvn -DskipTests clean package',
 '',
 '(You do not need to do this if you downloaded a 

In [60]:
readme_rdd.filter(lambda s: 'a' in s).collect()

['# Apache Spark',
 'Spark is a fast and general cluster computing system for Big Data. It provides',
 'high-level APIs in Scala, Java, Python, and R, and an optimized engine that',
 'supports general computation graphs for data analysis. It also supports a',
 'rich set of higher-level tools including Spark SQL for SQL and DataFrames,',
 'MLlib for machine learning, GraphX for graph processing,',
 'and Spark Streaming for stream processing.',
 '<http://spark.apache.org/>',
 '## Online Documentation',
 'You can find the latest Spark documentation, including a programming',
 'guide, on the [project web page](http://spark.apache.org/documentation.html).',
 'This README file only contains basic setup instructions.',
 '## Building Spark',
 'Spark is built using [Apache Maven](http://maven.apache.org/).',
 'To build Spark and its example programs, run:',
 '    build/mvn -DskipTests clean package',
 '(You do not need to do this if you downloaded a pre-built package.)',
 'You can build Spark u

In [65]:
readme_rdd.filter(lambda s: 'b' in s).collect()

['MLlib for machine learning, GraphX for graph processing,',
 'guide, on the [project web page](http://spark.apache.org/documentation.html).',
 'This README file only contains basic setup instructions.',
 'Spark is built using [Apache Maven](http://maven.apache.org/).',
 'To build Spark and its example programs, run:',
 '    build/mvn -DskipTests clean package',
 '(You do not need to do this if you downloaded a pre-built package.)',
 'You can build Spark using more than one thread by using the -T option with Maven, see ["Parallel builds in Maven 3"](https://cwiki.apache.org/confluence/display/MAVEN/Parallel+builds+in+Maven+3).',
 'More detailed documentation is available from the project site, at',
 '["Building Spark"](http://spark.apache.org/docs/latest/building-spark.html).',
 '    ./bin/spark-shell',
 '    ./bin/pyspark',
 'To run one of them, use `./bin/run-example <class> [params]`. For example:',
 '    ./bin/run-example SparkPi',
 'You can set the MASTER environment variable when

In [68]:
spark_readme = "file:///mnt/c/Users/bejao/OneDrive/data/spark-readme.md"
# rdd = rdd.repartition(40)
rdd = sc.textFile(spark_readme)
mots = rdd.flatMap( lambda line: line.split(" "))
mots_un = mots.map(lambda mot: (mot, 1))
count_mot = mots_un.reduceByKey( lambda un, last: un + last)
count_mot.collect()

# inverse = count_mot.map(lambda mot_count: (mot_count[1], mot_count[0]) )
# tri = inverse.sortByKey(ascending=False)
# filtre = tri.filter(lambda count_mot: len(count_mot[1]>3))
# filtre.collect()

[('#', 1),
 ('Apache', 1),
 ('Spark', 16),
 ('', 71),
 ('is', 6),
 ('It', 2),
 ('provides', 1),
 ('high-level', 1),
 ('APIs', 1),
 ('in', 6),
 ('Scala,', 1),
 ('Java,', 1),
 ('an', 4),
 ('optimized', 1),
 ('engine', 1),
 ('supports', 2),
 ('computation', 1),
 ('analysis.', 1),
 ('set', 2),
 ('of', 5),
 ('tools', 1),
 ('SQL', 2),
 ('MLlib', 1),
 ('machine', 1),
 ('learning,', 1),
 ('GraphX', 1),
 ('graph', 1),
 ('processing,', 1),
 ('Documentation', 1),
 ('latest', 1),
 ('programming', 1),
 ('guide,', 1),
 ('[project', 1),
 ('README', 1),
 ('only', 1),
 ('basic', 1),
 ('instructions.', 1),
 ('Building', 1),
 ('using', 5),
 ('[Apache', 1),
 ('run:', 1),
 ('do', 2),
 ('this', 1),
 ('downloaded', 1),
 ('more', 1),
 ('than', 1),
 ('-T', 1),
 ('Maven', 1),
 ('3"](https://cwiki.apache.org/confluence/display/MAVEN/Parallel+builds+in+Maven+3).',
  1),
 ('documentation', 3),
 ('project', 1),
 ('site,', 1),
 ('at', 2),
 ('Spark"](http://spark.apache.org/docs/latest/building-spark.html).', 1),
 ('

# `Parallelize`

* Q : Supp que ns avons 1 objet Python très volumineux et on souhaite y appliquer plusieurs transformations et actions sur Spark


In [69]:
liste = ['scala', 'java', 'hadoop', 'spark', 'akka', 'spark vs hadoop', 'pyspark', 'pyspark and spark']
liste_tuple1 = [('a', 7), ('a', 2), ('b', 2)]
liste_tuple2 = [('a', 2), ('d', 1), ('b', 1)]
liste_lazy = range(100)
liste_mixed = [("a", ["x", "y", "z"]), ("b", ["p", "r"])]


In [70]:
# Distribute a local Python collection to form an RDD
rdd = sc.parallelize(liste)
rdd1 = sc.parallelize(liste_tuple1)
rdd2 = sc.parallelize(liste_tuple2)
rdd3 = sc.parallelize(liste_lazy)
rdd4 = sc.parallelize(liste_mixed)

In [72]:
# Return a list with all RDD elements
rdd.collect()

# Count RDD instances
rdd2.count()

3

22/02/01 23:46:48 WARN spark.HeartbeatReceiver: Removing executor 0 with no recent heartbeats: 483021 ms exceeds timeout 120000 ms
22/02/01 23:46:50 ERROR scheduler.TaskSchedulerImpl: Lost executor 0 on 172.23.154.28: worker lost
22/02/01 23:46:50 WARN storage.BlockManagerMasterEndpoint: No more replicas available for broadcast_19_piece0 !
22/02/01 23:46:50 WARN storage.BlockManagerMasterEndpoint: No more replicas available for rdd_4_1 !
22/02/01 23:46:50 WARN storage.BlockManagerMasterEndpoint: No more replicas available for rdd_4_0 !
22/02/01 23:46:50 ERROR client.TransportClient: Failed to send RPC RPC 6619130823457919586 to /172.23.154.28:36674: java.io.IOException: Broken pipe
java.io.IOException: Broken pipe
	at sun.nio.ch.FileDispatcherImpl.write0(Native Method)
	at sun.nio.ch.SocketDispatcher.write(SocketDispatcher.java:47)
	at sun.nio.ch.IOUtil.writeFromNativeBuffer(IOUtil.java:93)
	at sun.nio.ch.IOUtil.write(IOUtil.java:65)
	at sun.nio.ch.SocketChannelImpl.write(SocketChannel

# Ex 1

* Q : Décompresser le fichier [purchases.gz](https://1drv.ms/u/s!AmJGbSlW18YGsbkXW0jFkWAWpznBeg?e=uh1W7o) et Copier le dans HDFS après avoir crée un dossier `/hadoop/data` ? 
* Le fichier purchases.txt ne comporte pas de header 
=> `| date | time | store name | item description | cost | method of payment |` 
* Q : Vérifiez la présence du fichier, sa taille en Mo et lire ses premières lignes (penser à utiliser l'option `more`) ?

    ```
    hdfs dfs -ls /
    hdfs dfs -mkdir -p /hadoop/data/purchases
    hdfs dfs -put /mnt/c/Users/bejao/OneDrive/data/purchases.txt /hadoop/data/purchases
    hdfs dfs -cat /hadoop/data/purchases/purchases.txt | more
    hdfs dfs -ls -h /hadoop/data/purchases

    ```

* Q : Lancer YARN et HIVE ? 
* Q : On cherche à calculer et à réccupérer le résultat en local : 
* du total des ventes par magasin ? 
* Quelle est le montant des ventes réalisé par le magasin Buffalo ? 
* Le total des ventes par type de produit ? 
* Quelle est la valeur des ventes pour la catégorie Toys ? Et pour la catégorie Consumer Electronics?
* L'achat le plus cher par magasin ?
* Quelle est la valeur de la vente la plus élévée pour les magasins suivants : o Reno, o Toledo, o Chandler ?
* Q :  Le nbre de transactions et la somme totale correspondante (tout magasin confondu) ; i.e le chiffre d'affaire ?    
* La moyenne des achats par jour ?   
    <br>

In [ ]:
!cat test.txt \
| python3 /mnt/c/Users/bejao/OneDrive/BigData/2-TP-MapReduce-Python-Yarn-Spark/scripts-MR/7-most-popular-page/mapper.py \
| sort \
| python3 /mnt/c/Users/bejao/OneDrive/BigData/2-TP-MapReduce-Python-Yarn-Spark/scripts-MR/7-most-popular-page/reducer.py

/ 	 14


# Ex 2

* Ex : Décompresser le fichier [access_log.gz](https://1drv.ms/u/s!AmJGbSlW18YGsbl7RzzPkT9v-qArTw?e=YA2zsE) et Copier le dans HDFS après avoir crée un dossier /`hadoop/data/access-log` ? Voici un descriptif du fichier log : 
    - Une ligne quelconque du log ressemble à :  
    `10.223.157.186 - - [15/Jul/2009:15:50:51 -0700] "GET /assets/css/960.css HTTP/1.1" 304 -`
    - nom des col corresp :  
    `ip, identity, username, datetime, tz, method, page, http_version, status, content_size`
    - Le format est donc le suivant :   
    `%h %l %u %t \"%r\" %>s %b`
* Avec : 
    - `%h` is the IP address of the client
    - `%l` is identity of the client, or `"-"` if it's unavailable
    - `%u` is username of the client, or `"-"` if it's unavailable
    - `%t` is the time that the server finished processing the request. The format is [`day/month/year:hour:minute:second zone`]
    - `%r` is the request line from the client is given (in double quotes). It contains the method, path, query-string, and protocol or the request.
    - `%>`s is the status code that the server sends back to the client. You will see see mostly status codes 200 (OK - The request has succeeded), 304 (Not Modified) and 404 (Not Found). See more information on status codes in W3C.org
    - `%b` is the size of the object returned to the client, in bytes. It will be `"-"` in case of status code 304.  
    <br>
1. Q : Vérifiez la présence du fichier, sa taille en Mo et lire ses premières lignes ?
2. Q : Lancer un job MR qui calcule le nb de clic par page ? Combien de hits a reçu la page `/assets/js/the-associates.js`
3. Q : Lancer un job MR qui calcule le nb de clic par adresse IP ? Combien de fois l'adresse IP 10.99.99.186 a été sollicité ? 
4. Q : Lancer un job MR pour trouver la page la plus visitée et le nombre de visites correspondant ? 

# Ex 3

* Décompresser le fichier `forum_data.tar.gz` et Copier le dans HDFS après avoir crée un dossier /`hadoop/data/forum-data` ?
1. Q : Lancer un job MR qui retourne les 10 posts les plus longs triés dans un ordre croissant de longueur ?
2. Q : Lancer un job MR qui calcule le nb de clic par adresse IP ?
3. Q : Lancer un job MR pour trouver la page la plus visitée et le nombre de visites correspondant ? 